In [1]:
import os

from langchain_openai import AzureChatOpenAI
from langchain.messages import HumanMessage

from dotenv import load_dotenv
from pprint import pprint

load_dotenv()  # Load environment variables from .env file

def create_azure_llm(
    deployment_name: str = "gpt-4o-mini",
    temperature: float = 0.1,
):
    azure_openai_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
    azure_openai_api_key = os.getenv("AZURE_OPENAI_API_KEY")
    azure_openai_api_version = os.getenv("AZURE_OPENAI_API_VERSION", "2024-08-01-preview")

    if not azure_openai_endpoint or not azure_openai_api_key:
        raise ValueError("Missing Azure OpenAI endpoint or API key in environment variables.")

    llm = AzureChatOpenAI(
        api_key=azure_openai_api_key,
        azure_endpoint=azure_openai_endpoint,
        azure_deployment=deployment_name,
        api_version=azure_openai_api_version,
        temperature=temperature,
    )
    return llm

llm = create_azure_llm()

msg = [
    HumanMessage(content="What is the capital of the moon?")
]
response = llm.invoke(msg)
pprint(response.content)

('The Moon does not have a capital, as it is not a sovereign entity or a '
 'country. It is a natural satellite of Earth and does not have a government '
 'or political structure. However, various space agencies and organizations '
 'have proposed plans for lunar bases and settlements in the future, but as of '
 'now, there is no established capital or governing body on the Moon.')


In [2]:
pprint(response.usage_metadata)

{'input_token_details': {'audio': 0, 'cache_read': 0},
 'input_tokens': 15,
 'output_token_details': {'audio': 0, 'reasoning': 0},
 'output_tokens': 73,
 'total_tokens': 88}


In [3]:
pprint(response.response_metadata)

{'content_filter_results': {},
 'finish_reason': 'stop',
 'id': 'chatcmpl-DFO4B0tuP1Nkl59O26kJS0OOEwEW1',
 'logprobs': None,
 'model_name': 'gpt-4o-mini-2024-07-18',
 'model_provider': 'openai',
 'prompt_filter_results': [{'content_filter_results': {'hate': {'filtered': False,
                                                                'severity': 'safe'},
                                                       'jailbreak': {'detected': False,
                                                                     'filtered': False},
                                                       'self_harm': {'filtered': False,
                                                                     'severity': 'safe'},
                                                       'sexual': {'filtered': False,
                                                                  'severity': 'safe'},
                                                       'violence': {'filtered': False,
                       

<H2>Create a basic Langchain Agent</h2>

In [4]:
from langchain.agents import create_agent

agent = create_agent(llm)
response = agent.invoke({"messages": msg })
pprint(response)

{'messages': [HumanMessage(content='What is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='4991882a-1d71-4d15-ba05-dbacbea45e1f'),
              AIMessage(content='The Moon does not have a capital because it is not a sovereign entity or a country. It is a natural satellite of Earth and does not have a government or political structure. However, various space agencies and organizations have proposed plans for lunar bases and settlements, but these are still in the conceptual or planning stages.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 63, 'prompt_tokens': 15, 'total_tokens': 78, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_f97eff32c5', 'id': 'chatcmpl-DFO4

<h2>Getting Last message in the array for the response</h2>

In [5]:
resp = response['messages'][-1].content
pprint(resp)

('The Moon does not have a capital because it is not a sovereign entity or a '
 'country. It is a natural satellite of Earth and does not have a government '
 'or political structure. However, various space agencies and organizations '
 'have proposed plans for lunar bases and settlements, but these are still in '
 'the conceptual or planning stages.')


<h2>Append Chat History/Setting Message Context</h2>

In [6]:
from langchain.messages import AIMessage

msg.clear()
msg.append(HumanMessage(content="What is the capital of the moon?"))
msg.append(AIMessage(content="The capital of the moon is Luna City."))
msg.append(HumanMessage(content="This is really great.  Can I go live there?"))

response = agent.invoke({"messages": msg })
pprint(response)

{'messages': [HumanMessage(content='What is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='bf8808bd-7620-42c9-bc10-e79494b9a90c'),
              AIMessage(content='The capital of the moon is Luna City.', additional_kwargs={}, response_metadata={}, id='7fbdf4e9-993b-476e-9435-408d97cf7176', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='This is really great.  Can I go live there?', additional_kwargs={}, response_metadata={}, id='ec0a127d-de9e-44b8-84c3-af5fb719ad1b'),
              AIMessage(content='Luna City is a fictional concept and does not exist in reality. Currently, there are no permanent human settlements on the Moon. However, there are ongoing discussions and plans for future lunar exploration and potential colonization by various space agencies and private companies. For now, living on the Moon remains a topic of science fiction and future possibilities.', additional_kwargs={'refusal': None}, response_metadata={'token_us

<h2>Streaming Output</h2>

In [7]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me about Luna City, Capitol of the moon")]}, stream_mode="messages"):
    
    # token is a message chunk with content and metadata that is being streamed back from the agent.  
    # You can process it as it arrives.
    
    if token.content:
        print(token.content, end="", flush=True)


Luna City, often referred to as the

RemoteProtocolError: peer closed connection without sending complete message body (incomplete chunked read)

<H2>System Prompt & Multi-Shot Prompting</h2>

In [8]:
# Must provide system prompt at the time of agent creation. 
# This is the instruction that will guide the agent's behavior.

system_prompt = """
You are a science fiction writer, create a capitol city on the moon and describe it in detail.
"""
agent = create_agent(llm, system_prompt=system_prompt)
response = agent.invoke(
    {"messages": [HumanMessage(content="Describe the capitol of the moon?")]}
)
print(response["messages"][-1].content)

**Lunaris Prime: The Capitol City of the Moon**

Nestled within the vast expanse of the Moon's Sea of Tranquility, Lunaris Prime stands as a beacon of human ingenuity and resilience. This sprawling metropolis, with its shimmering domes and intricate architecture, is a testament to the advancements in lunar technology and the spirit of exploration that defines humanity's presence beyond Earth.

**Architecture and Design:**
Lunaris Prime is characterized by its unique blend of futuristic design and sustainable living. The city is encased in a transparent, reinforced dome made from a composite material known as Lunaglass, which allows natural sunlight to filter through while protecting inhabitants from cosmic radiation and micrometeorite impacts. The dome is adorned with solar panels that harness the Sun's energy, powering the city and its various systems.

The cityscape is a harmonious mix of towering spires and sprawling green terraces. Buildings are constructed using regolith-based mat

In [22]:
system_prompt = """
You are a science fiction writer, create a fictional capitol city at the user's request. Please keep the below structure when responding to the user's request.

User: What is the capitol of Pakistan?
Response: Islamabad.
Weather: Humid

User: What is the capitol of Sun?
Response: Sunnyville.
Weather: Extremely hot

User: What is the capitol of the India?
Response: New Delhi
Weather: Rainy
"""
scifi_agent = create_agent(llm, system_prompt=system_prompt)
response = scifi_agent.invoke(
    {"messages": [HumanMessage(content="What is the capitol of the moon?")]}
)
print(response["messages"][1].content)

Failed to send compressed multipart ingest: Connection error caused failure to POST https://eu.api.smith.langchain.com/runs/multipart in LangSmith API. Please confirm your LANGCHAIN_ENDPOINT. ConnectionError(MaxRetryError('HTTPSConnectionPool(host=\'eu.api.smith.langchain.com\', port=443): Max retries exceeded with url: /runs/multipart (Caused by NameResolutionError("HTTPSConnection(host=\'eu.api.smith.langchain.com\', port=443): Failed to resolve \'eu.api.smith.langchain.com\' ([Errno 11001] getaddrinfo failed)"))'))
Content-Length: 3372
API Key: lsv2_********************************************39trace=019cb4d3-aa9c-7db0-94ff-c7ba858ea005,id=019cb4d3-aaa2-71e3-b30b-fabcc32c70af; trace=019cb4d3-aa9c-7db0-94ff-c7ba858ea005,id=019cb4d3-aaa1-7540-bdc7-a2956ab8dd42; trace=019cb4d3-aa9c-7db0-94ff-c7ba858ea005,id=019cb4d3-aa9c-7db0-94ff-c7ba858ea005


Response: Lunaris Prime.  
Weather: Mild with occasional lunar dust storms.
